# Push Sprint Further: Unfolding Modulation
**Runtime → A100 or T4**

Sprint gets 4.8x but overfits after step 1100. Marathon never overfits but
is slower to converge. Can we push Sprint further by:
1. Re-extracting spectral targets as the model trains (unfolding)
2. Pushing the marathon's anti-overfit property into Sprint's fast convergence

The synthesizer feedback loop: the model's output feeds back into
its own modulation signal.

In [ ]:
# Cell 1: Setup
import os, subprocess
if os.path.exists('/content/nanogpt-prism'):
    subprocess.run(['rm', '-rf', '/content/nanogpt-prism'])
!git clone https://github.com/timepointai/nanogpt-prism-shakespeare.git /content/nanogpt-prism
%cd /content/nanogpt-prism
!pip install -q transformers tiktoken datasets
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
!python data/shakespeare_char/prepare.py
print('Ready.')

In [ ]:
# Cell 2: Train teacher + extract (skip if cached)
import os, subprocess
if os.path.exists('.prism_cache/shakespeare/spectra.json'):
    print('Cached.')
else:
    subprocess.run(['python', 'train.py', 'config/train_shakespeare_char.py',
        '--max_iters=2000', '--eval_interval=1000', '--eval_iters=50',
        '--log_interval=1000', '--out_dir=out-teacher',
        '--always_save_checkpoint=True', '--compile=False',
        '--prism_init=False', '--wandb_log=False'], timeout=600)
    subprocess.run(['python', 'prism_extract.py',
        '--ckpt', 'out-teacher/ckpt.pt',
        '--out', '.prism_cache/shakespeare'], timeout=120)
    print('Done.')

In [ ]:
# Cell 3: Sprint push experiments
import subprocess, time, re, json, os

STEPS = 5000
EVAL = 100

P = [
    '--prism_init=True', '--prism_align=0.75',
    '--prism_spectra=.prism_cache/shakespeare/spectra.json',
    '--prism_directions=.prism_cache/shakespeare/directions.pt',
]

configs = [
    # Controls from v2
    ('baseline', ['--prism_init=False']),
    ('sprint', P + ['--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.005', '--prism_mod_decay=0.999']),
    ('marathon', P + ['--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.9999']),
    
    # Sprint + unfolding (re-extract targets as model trains)
    ('sprint_unfold_200', P + ['--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.005', '--prism_mod_decay=0.999',
        '--prism_unfold=200']),
    ('sprint_unfold_100', P + ['--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.005', '--prism_mod_decay=0.999',
        '--prism_unfold=100']),
    
    # Sprint speed + marathon anti-overfit: gentle mod but with unfolding
    ('sprint_sustained_unfold', P + ['--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.005', '--prism_mod_decay=0.999',
        '--prism_mod_sustain=0.01', '--prism_mod_sustain_decay=0.9999',
        '--prism_mod_transition=500', '--prism_unfold=200']),
    
    # Marathon + unfolding (does unfolding help marathon too?)
    ('marathon_unfold_200', P + ['--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.9999',
        '--prism_unfold=200']),
    
    # Higher mod strength with unfolding (can we be more aggressive?)
    ('sprint_hot_unfold', P + ['--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.999',
        '--prism_unfold=100']),
]

all_curves = {}

for name, extra in configs:
    log_path = f'/content/push_{name}.txt'
    if os.path.exists(log_path):
        print(f'[{name}] SKIP')
        with open(log_path) as f:
            stdout = f.read()
    else:
        print(f'\n{"="*60}')
        print(f'  {name}')
        print(f'{"="*60}')
        t0 = time.time()
        result = subprocess.run(
            ['python', 'train.py', 'config/train_shakespeare_char.py',
             f'--max_iters={STEPS}', f'--eval_interval={EVAL}',
             '--eval_iters=50', '--log_interval=100',
             f'--out_dir=out-push-{name}',
             '--wandb_log=False', '--compile=False'] + extra,
            capture_output=True, text=True, timeout=1200
        )
        stdout = result.stdout
        with open(log_path, 'w') as f:
            f.write(stdout)
        if result.returncode != 0:
            print(f'ERROR: {result.stderr[-500:]}')
        print(f'  Wall: {time.time()-t0:.0f}s')

    val = {}
    for line in stdout.split('\n'):
        m = re.search(r'step (\d+): train loss ([\d.]+), val loss ([\d.]+)', line)
        if m:
            val[int(m.group(1))] = float(m.group(3))
    all_curves[name] = val
    if val:
        best_v = min(val.values())
        best_s = min(val, key=val.get)
        print(f'  [{name}] best: {best_v:.4f} @ step {best_s}')

with open('/content/push_curves.json', 'w') as f:
    json.dump({k: {str(s): v for s, v in c.items()} for k, c in all_curves.items()}, f, indent=2)
print('\nAll saved.')

In [ ]:
# Cell 4: Results
import json

curves = {k: {int(s): v for s, v in c.items()}
          for k, c in json.load(open('/content/push_curves.json')).items()}

bb = min(curves['baseline'].values())
bs = min(curves['baseline'], key=curves['baseline'].get)

print('='*75)
print(f'  TARGET: val_loss <= {bb:.4f} (baseline best at step {bs})')
print('='*75)

print(f'\n{"Runner":>25s}  {"Hit@":>6s}  {"Speed":>6s}  {"Best":>8s}  {"@step":>6s}  {"@5000":>8s}')
print('-' * 65)
for name in sorted(curves.keys(), key=lambda n: min(curves[n].values()) if curves[n] else 999):
    c = curves[name]
    if not c: continue
    best_v = min(c.values())
    best_s = min(c, key=c.get)
    at_5k = c.get(5000, c.get(max(c.keys()), 0))
    hit = None
    for s in sorted(c.keys()):
        if c[s] <= bb:
            hit = s
            break
    spd = f'{bs/hit:.1f}x' if hit else '—'
    hit_str = str(hit) if hit else 'never'
    print(f'{name:>25s}  {hit_str:>6s}  {spd:>6s}  {best_v:>8.4f}  {best_s:>6d}  {at_5k:>8.4f}')

# Key steps
print(f'\n--- Key steps ---')
names = list(curves.keys())
print(f'{"Step":>6s}', end='')
for n in names: print(f'  {n[:10]:>10s}', end='')
print()
for step in [0, 200, 400, 600, 800, 1000, 1500, 2000, 3000, 4000, 5000]:
    vals = {n: curves[n].get(step) for n in names}
    active = [v for v in vals.values() if v]
    if not active: continue
    best = min(active)
    print(f'{step:>6d}', end='')
    for n in names:
        v = vals[n]
        if v:
            m = '*' if abs(v-best)<0.002 else ' '
            print(f'  {v:>9.4f}{m}', end='')
        else:
            print(f'  {"":>10s}', end='')
    print()

In [ ]:
# Cell 5: Download
from google.colab import files
files.download('/content/push_curves.json')
for n in ['baseline','sprint','marathon','sprint_unfold_200','sprint_unfold_100',
          'sprint_sustained_unfold','marathon_unfold_200','sprint_hot_unfold']:
    try: files.download(f'/content/push_{n}.txt')
    except: pass